# QueryMind — Training Analysis

Visualize reward curves, action distributions, and plan comparisons.

Run this after training to analyze agent performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "Inter"

## 1. Load Training Logs

In [ ]:
# Load per-query performance from training callback
df = pd.read_csv("../logs/query_performance.csv")
print(f"Queries: {df['query_id'].nunique()}")
print(f"Total episodes: {df['n_episodes'].sum()}")
df.head(10)

## 2. Reward Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean latency ratio by query
ax = axes[0]
df_sorted = df.sort_values("mean_ratio", ascending=False)
colors = ["#00C853" if r > 1.0 else "#FF1744" for r in df_sorted["mean_ratio"]]
ax.barh(df_sorted["query_id"], df_sorted["mean_ratio"], color=colors)
ax.axvline(x=1.0, color="white", linestyle="--", alpha=0.7)
ax.set_xlabel("Latency Ratio (>1 = agent wins)")
ax.set_title("Mean Speedup by Query")

# Win rate by query
ax = axes[1]
ax.barh(df_sorted["query_id"], df_sorted["win_rate"] * 100, color="#2979FF")
ax.set_xlabel("Win Rate (%)")
ax.set_title("Agent Win Rate by Query")

plt.tight_layout()
plt.savefig("../reports/query_performance.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Benchmark Results

In [ ]:
# Load benchmark CSV (after running querymind-bench)
try:
    bench = pd.read_csv("../reports/benchmark_results.csv")

    pivot = bench.pivot_table(
        index="query_id", columns="config", values="latency_ms", aggfunc="median"
    )

    print("Benchmark Results (median latency, ms):")
    print(pivot.round(1))

    if "querymind" in pivot.columns and "pg_default" in pivot.columns:
        speedups = pivot["pg_default"] / pivot["querymind"]
        print(f"\nGeometric mean speedup: {np.exp(np.mean(np.log(speedups))):.3f}x")
        print(f"Win rate: {(speedups > 1.0).mean():.1%}")
except FileNotFoundError:
    print("Run querymind-bench first to generate benchmark results.")